### E-Commerce US Dataset -- Feature Engineering and validation

\Importing Necessary Libraries

In [1]:
import numpy as np
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

\Loading the processed data

In [2]:
path = r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\processed"

master = pd.read_csv(os.path.join(path,"master_clean.csv"))
items = pd.read_csv(os.path.join(path,"items_clean.csv"))

In [3]:
master.shape

(10000, 39)

In [4]:
items.shape

(21838, 23)

\Date data type conversion

In [5]:
for col in ["order_purchase_timestamp","order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    if col in master.columns:
        master[col] = pd.to_datetime(master[col],errors = "coerce")

In [6]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"],errors="coerce")

### Feature Engineering

\Delivery Time Metrics

In [7]:
#delivery days
master["delivery_days"] = (master["order_delivered_customer_date"] - master["order_purchase_timestamp"]).dt.days

#estimated delivery days
master["estimated_days"] = (master["order_estimated_delivery_date"] - master["order_purchase_timestamp"]).dt.days

#delivery delay
master["delivery_delay"] = master["delivery_days"] - master["estimated_days"]

#late flag
master["is_late"] = (master["delivery_delay"]> 0).astype(int)

#Delivery Speed Category
master["delivery_speed"] = pd.cut(master["delivery_days"],bins=[0,5,10,100],labels=["Fast","Medium","Slow"])

\Average Order Value + Payment Behavior

In [9]:
# avg value per item
master["avg_item_value"] = (master["total_price"]/master["n_items"]).round(2)

#Freight ratio
master["freight_ratio"] = (master["total_freight"]/ (master["total_price"] + 0.01) *100).round(2)

#payment behavior
master["uses_installments"] = (master["max_installments"]>1).astype(int)
master["installment_category"] = pd.cut(master["max_installments"],bins=[0,1,6,100],
                                        labels=["Single","Short-term","Long-term"])

\Profit feature

In [10]:
items["profit"] = (items["price"] - items["cost"]).round(2)
items["profit_margin_pct"] = ((items["profit"] / items["price"]) * 100).round(2)

\Customer Life Metrics

In [11]:
#customer table
customer_features = master.groupby("customer_unique_id").agg(
    total_orders     = ("order_id", "count"),
    total_spend      = ("total_payment", "sum"),
    avg_order_value  = ("total_payment", "mean"),
    total_items      = ("n_items", "sum"),
    avg_review_given = ("avg_review_score", "mean"),
    first_order      = ("order_purchase_timestamp", "min"),
    last_order       = ("order_purchase_timestamp", "max"),
).reset_index()

# Repeat customer
customer_features["is_repeat_customer"] = (customer_features["total_orders"] > 1).astype(int)

# Customer tenure 
customer_features["tenure_days"] = (customer_features["last_order"] -
                                     customer_features["first_order"]).dt.days

# Customer value tier
customer_features["value_tier"] = pd.qcut(customer_features["total_spend"],
                                           q=4, labels=["Bronze","Silver","Gold","Platinum"])

\Customer Engagement

In [13]:
customer_features["review_rate"] = master.groupby("customer_unique_id")["has_review"].mean().values
orders_norm = customer_features["total_orders"] / customer_features["total_orders"].max()
spend_norm  = customer_features["total_spend"]  / customer_features["total_spend"].max()
customer_features["engagement_score"] = (
    orders_norm * 0.4 + customer_features["review_rate"] * 0.3 + spend_norm * 0.3
).round(3)

In [14]:
customer_features["engagement_score"].describe()

count    7898.000000
mean        0.393631
std         0.088765
min         0.067000
25%         0.374000
50%         0.389000
75%         0.420000
max         0.900000
Name: engagement_score, dtype: float64

\Seller Performance Metrics

In [15]:
#seller Table
seller_features = items.groupby("seller_id").agg(
    total_revenue = ("price","sum"),
    total_items_sold = ("order_id","count"),
    avg_price = ("price","mean"),
    n_unique_orders = ("order_id","nunique"),
    n_products = ("product_id","nunique")
).reset_index()

#revenue per order
seller_features["revenue_per_order"] = (seller_features["total_revenue"]/seller_features["n_unique_orders"]).round(2)

#seller tier
seller_features["seller_tier"] = pd.qcut(seller_features["total_revenue"],q=3,labels=["Low","Mid","Top"])


\Product Popularity Metrics

In [16]:
#product table
product_features = items.groupby("product_id").agg(
    times_ordered = ("order_id", "count"),
    total_revenue = ("price", "sum"),
    avg_price = ("price", "mean")
).reset_index()

#product popularity tier
product_features["popularity_tier"] = pd.qcut(product_features["times_ordered"],
                                               q=3, labels=["Low","Medium","High"],
                                               duplicates="drop")

#revenue contribution rank
product_features["product_revenue_rank"] = product_features["total_revenue"].rank(ascending=False).astype(int)

\Saving the processed + feature added files 

In [17]:
out_path = r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\processed-feature"
master.to_csv(os.path.join(out_path, "master_features.csv"), index=False)
customer_features.to_csv(os.path.join(out_path, "customer_features.csv"), index=False)
seller_features.to_csv(os.path.join(out_path, "seller_features.csv"), index=False)
product_features.to_csv(os.path.join(out_path, "product_features.csv"), index=False)

### Data Validation

In [20]:
class DataValidator:
    def __init__(self, df, name="dataset"):
        self.df = df             
        self.name = name         
        self.results = []        

    def _record(self, category, rule, passed, detail=""):
        """Internal - oru check result-ah store pannu."""
        self.results.append({
            "category": category,
            "rule": rule,
            "status": "PASS" if passed else "FAIL",
            "detail": detail
        })
    

In [21]:
#SCHEMA VALIDATION
def validate_schema(self, expected_columns):
        """Ellaa expected columns-um irukaa?"""
        actual = set(self.df.columns)
        missing = set(expected_columns) - actual
        passed = len(missing) == 0
        self._record("Schema", "All expected columns present", passed,
                     f"Missing: {missing}" if missing else "OK")
        return self